# Train and Evaluate Detection Model

Train and evaluate the YOLO26s object detection model.

**Workflow**

1. Set `PRESET = "smoke"` and run all sections to verify the training and evaluation pipeline.
2. Set `PRESET = "development"` and run all sections to train the development model.
3. Review the training history, evaluation metrics and prediction visualizations.

To inspect an existing training run, set `DATASET_NAME` and `RUN_NAME` accordingly and skip training.


## 1. Setup

Use the corresponding experiment's Python environment as the notebook kernel.


In [ ]:

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from ultralytics import YOLO

from grid_detection.evaluation import evaluate_run
from grid_detection.research import DATA_DIR, PROJECT_ROOT, RUNS_DIR
from grid_detection.training import (load_saved_run, prepare_training_options, resolve_device,
                                     train_model, validate_directory_name)
from grid_detection.visualize import visualize_predictions
from grid_detection.notebook_display import show_table


## 2. Configuration

Set `DATASET_NAME` to select the input dataset folder and `RUN_NAME` to select the output run folder. Use `DEVICE` to select the CPU or an accelerator, or `auto` for automatic selection.


In [ ]:
PRESET = "smoke"       # "smoke", "development"
DATASET_NAME = PRESET
RUN_NAME = PRESET
DEVICE = "auto"        # "auto", "cpu", "mps", "cuda", "cuda:0"

presets = {
    "smoke": {
        "epochs": 2,
    },
    "development": {
        "epochs": 100,
    },
}
if PRESET not in presets:
    raise ValueError(f"Unknown preset: {PRESET}")
for label, name in (("dataset", DATASET_NAME), ("run", RUN_NAME)):
    validate_directory_name(name, label)

device = resolve_device(DEVICE)
model_name = "yolo26s.pt"
model_path = PROJECT_ROOT / "weights" / model_name
training = dict(
    imgsz=1280,
    batch=8,
    patience=20,
    workers=4,
    optimizer="auto",
    seed=42,
    deterministic=True,
    pretrained=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    translate=0.05,
    scale=0.1,
    degrees=0.0,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.0,
    bgr=0.0,
    mosaic=0.0,
    mixup=0.0,
    cutmix=0.0,
    copy_paste=0.0,
    plots=True,
    val=True,
    save=True,
    **presets[PRESET],
)
training = prepare_training_options(training, device)
print(f"Preset: {PRESET}; model: {model_name}; epochs: {training['epochs']}; batch: {training['batch']}; image size: {training['imgsz']}")
print(f"Requested device: {DEVICE}; selected device: {device}; requested AMP: {training['amp']}")


## 3. Train model

Initialize YOLO26s with pretrained weights and fine-tune it on the selected dataset. An existing run folder is protected from accidental overwrite.


In [ ]:
train_model(dataset_name=DATASET_NAME, run_name=RUN_NAME, preset=PRESET,
            requested_device=DEVICE, device=device, model_path=model_path, training=training,
            data_dir=DATA_DIR, runs_dir=RUNS_DIR, project_root=PROJECT_ROOT)


## 4. Load saved run

The following analysis uses the saved training configuration, history and checkpoints rather than assuming the current training settings were used.


In [ ]:
run = load_saved_run(RUNS_DIR, RUN_NAME)
inspect_dir, saved, saved_args = run.directory, run.metadata, run.args
saved_root, saved_config, saved_records, saved_yaml = run.dataset_root, run.dataset_config, run.records, run.data_yaml
best_path, last_path, history = run.best_path, run.last_path, run.history
print(f"Run: {inspect_dir}")
print(f"Dataset: {saved_root}")
print(f"Best checkpoint: {best_path} ({'available' if best_path.is_file() else 'missing'})")
print(f"Last checkpoint: {last_path} ({'available' if last_path.is_file() else 'missing'})")
print(f"Recorded epochs: {len(history)}; last epoch: {history[-1]['epoch']}")


## 5. Analyze training history

Follow loss, precision, recall, mAP and learning-rate progression across epochs. Compare training and validation loss trends to assess convergence and possible overfitting.


In [ ]:
epochs = [float(row["epoch"]) for row in history]
groups = {
    "Training and validation losses": [k for k in history[0] if k.startswith(("train/", "val/")) and "loss" in k],
    "Precision and recall": [k for k in history[0] if k.startswith(("metrics/precision", "metrics/recall"))],
    "mAP50 and mAP50–95": [k for k in history[0] if k.startswith(("metrics/mAP50(", "metrics/mAP50-95("))],
    "Learning rates": [k for k in history[0] if k.startswith("lr/")],
}
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (title, keys) in zip(axes.flat, groups.items()):
    for key in keys:
        ax.plot(epochs, [float(row[key]) for row in history], marker="o", markersize=3, label=key)
    ax.set(title=title, xlabel="Epoch")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=8))
    if keys:
        ax.legend(fontsize=8)
for ax in (axes.flat[1], axes.flat[2]):
    ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1, decimals=2))
fig.tight_layout()
plt.show()
for name in ("confusion_matrix.png", "confusion_matrix_normalized.png", "BoxPR_curve.png", "BoxP_curve.png", "BoxR_curve.png"):
    path = inspect_dir / name
    if path.is_file():
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.imshow(plt.imread(path))
        ax.set_title(name)
        ax.axis("off")
        plt.show()


## 6. Analyze training performance

The saved run's reported elapsed training time and environment describe its computational cost and execution context. Compare requested and effective device and AMP settings when interpreting the results.


In [ ]:
if history[-1].get("time"):
    print(f"Reported elapsed training time: {float(history[-1]['time']):.1f} s")
else:
    print("Reported elapsed training time: —")
env = saved.get("provenance", {}).get("environment") or {}
versions = saved.get("versions") or {}
def available(value):
    return "—" if value is None else value
print(f"Saved training environment: Python {available(env.get('python'))} | PyTorch {available(versions.get('pytorch'))} | Ultralytics {available(versions.get('ultralytics'))}")
print(f"Device: requested={saved.get('device_requested')}, effective={saved.get('resolved_device')}; AMP: requested={saved.get('amp_requested')}, effective={saved.get('resolved_amp')}; workers={saved.get('resolved_workers')}")
print("Training throughput and peak memory: — in saved run metadata")

print(f"Model parameters: {sum(p.numel() for p in YOLO(str(best_path)).model.parameters()):,}" if best_path.is_file() else "Model parameters: — (best checkpoint missing)")
print(f"Completed epochs: {len(history)}")


## 7. Evaluate model

Evaluate the best checkpoint on the validation and test datasets. Precision, recall and mAP summarize detection quality; compare per-class AP to identify weaker classes.


In [ ]:
reports = evaluate_run(run, device=device)
splits_to_compare = ("val", "test")
def percent(value):
    return "—" if value is None else f"{value:.2%}"
show_table("Aggregate metrics", ("Metric", "Validation", "Test"),
           [(label, *(percent(reports[s]["aggregate"].get(key)) for s in splits_to_compare))
            for key, label in (("precision", "Precision"), ("recall", "Recall"), ("map50", "mAP50"), ("map75", "mAP75"), ("map50_95", "mAP50–95"))])
classes = list(dict.fromkeys(name for s in splits_to_compare for name in reports[s]["per_class"]))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (key, title) in zip(axes, (("ap50", "AP50"), ("ap75", "AP75"), ("ap50_95", "AP50–95"))):
    for i, (split, label, color) in enumerate((("val", "Validation", "#648bc5"), ("test", "Test", "#54a68b"))):
        for x, name in enumerate(classes):
            value = reports[split]["per_class"].get(name, {}).get(key)
            if value is not None:
                ax.bar(x + (i-.5)*.36, value*100, .36, color=color, label=label if x == 0 else None)
            else:
                ax.text(x + (i-.5)*.36, 2, "—", ha="center")
    ax.set(title=title, ylabel="AP (%)", ylim=(0, 100))
    ax.set_xticks(range(len(classes)), classes, rotation=35, ha="right")
from matplotlib.patches import Patch
fig.legend(handles=[Patch(color="#648bc5", label="Validation"), Patch(color="#54a68b", label="Test")], loc="upper center", ncol=2)
fig.tight_layout(rect=(0, 0, 1, .93))
plt.show()
stages = list(dict.fromkeys(k for s in splits_to_compare for k in reports[s]["timing_ms_per_image"]))
show_table("Inference timing (ms/image)", ("Stage", "Validation", "Test"),
           [(stage, *(f"{v:.2f}" if (v := reports[s]["timing_ms_per_image"].get(stage)) is not None else "—" for s in splits_to_compare)) for stage in stages])


## 8. Visualize predictions

Compare predicted bounding boxes and classes with ground truth in representative examples and selected failures, including missed detections and false positives. The failure scan limit controls how many samples are searched for those cases.


In [ ]:
DIAGNOSTIC_SPLIT = "val"
CONFIDENCE = 0.25
MATCH_IOU = 0.5
FAILURE_SCAN_LIMIT = 40
RANDOM_EXAMPLES = 3
visualize_predictions(saved_records, saved_root, saved_args, best_path, device,
                      DIAGNOSTIC_SPLIT, CONFIDENCE, MATCH_IOU, FAILURE_SCAN_LIMIT, RANDOM_EXAMPLES)
